In [1]:
import os
import re
import json
import time
import requests

from bs4 import BeautifulSoup

from urllib.parse import (
    urljoin,
    urlparse,
    urlunparse
)

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
#BLOQUE 2 (Ruta del JSON en directorio en el que estamos)
NOMBRE_PROGRAMA = "Extrae_Innovacion.ipynb"


ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:

        ruta_programa = root

        break


if ruta_programa is None:

    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(

    ruta_programa,

    "JSONs"

)

os.makedirs(

    CARPETA_JSON,

    exist_ok=True

)

In [3]:
# ==========================================================
# BLOQUE 3. CONFIGURACIÓN DE LA FUENTE
#
# Fuente: UPV Innovación
#
# Este bloque define:
# - URL de la página raíz
# - nombre y ruta del JSON
# - secciones semánticas de la página
# - cabeceras HTTP
# - funciones auxiliares de normalización
# - estructura base que tendrá el JSON
#
# La carpeta CARPETA_JSON se calcula previamente.
# ==========================================================

import os
import json
import re
import unicodedata
from urllib.parse import urljoin, urlparse


# ==========================================================
# 1. CONFIGURACIÓN GENERAL
# ==========================================================

URL_RAIZ = "https://innovacion.upv.es/"

NOMBRE_JSON = "innovacion.json"

RUTA_JSON = os.path.join(
    CARPETA_JSON,
    NOMBRE_JSON
)


# ==========================================================
# 2. SECCIONES SEMÁNTICAS
#
# Estas categorías no representan clases CSS del HTML.
# Representan la organización lógica que queremos conservar
# en el JSON.
# ==========================================================

SECCIONES_VALIDAS = {
    "Presentación",
    "Servicios de Innovación",
    "Iniciativas",
    "Historias de Innovación",
    "Temas",
    "Explora UPV",
    "Destacados"
}


# ==========================================================
# 3. CABECERAS HTTP
# ==========================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,image/avif,image/webp,"
        "*/*;q=0.8"
    ),
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Connection": "keep-alive"
}


# ==========================================================
# 4. FUNCIONES AUXILIARES
# ==========================================================

def limpiar_texto(texto):
    """
    Limpia y normaliza un fragmento de texto extraído
    del HTML.

    El objetivo es eliminar espacios y saltos de línea
    generados por la estructura HTML sin alterar el
    contenido textual.
    """

    if texto is None:
        return ""

    texto = str(texto)

    # Sustituir espacios, saltos de línea y tabulaciones
    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()


# ----------------------------------------------------------
# Normalización de nombres para identificadores
# ----------------------------------------------------------

def normalizar_identificador(texto):
    """
    Convierte un texto en un identificador estable.

    Ejemplo:
        'Historias de Innovación'
        -> 'historias_de_innovacion'
    """

    texto = limpiar_texto(texto)

    # Eliminar acentos
    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    # Minúsculas
    texto = texto.lower()

    # Sustituir caracteres no alfanuméricos
    texto = re.sub(
        r"[^a-z0-9]+",
        "_",
        texto
    )

    return texto.strip("_")


# ----------------------------------------------------------
# Convertir URL relativa en absoluta
# ----------------------------------------------------------

def normalizar_url(url, url_base=URL_RAIZ):
    """
    Convierte una URL relativa en absoluta utilizando
    la URL de la página desde la que se ha obtenido.
    """

    if not url:
        return ""

    url = url.strip()

    return urljoin(
        url_base,
        url
    )


# ----------------------------------------------------------
# Comprobar URL HTTP/HTTPS
# ----------------------------------------------------------

def es_url_valida(url):
    """
    Comprueba si una URL corresponde a HTTP o HTTPS.
    """

    if not url:
        return False

    try:

        analisis = urlparse(url)

        return analisis.scheme in {
            "http",
            "https"
        }

    except Exception:

        return False


# ----------------------------------------------------------
# Deduplicar elementos conservando el orden
# ----------------------------------------------------------

def deduplicar_lista(elementos, clave="url"):
    """
    Elimina elementos duplicados de una lista de diccionarios.

    Si se proporciona una clave, la comparación se realiza
    utilizando su valor.

    Se conserva la primera aparición.
    """

    resultado = []
    vistos = set()

    for elemento in elementos:

        if not isinstance(
            elemento,
            dict
        ):
            continue

        valor = elemento.get(
            clave,
            ""
        )

        if valor in vistos:
            continue

        vistos.add(valor)

        resultado.append(
            elemento
        )

    return resultado


# ----------------------------------------------------------
# Crear estructura vacía de una sección
# ----------------------------------------------------------

def crear_seccion(
    titulo,
    tipo="seccion"
):
    """
    Crea la estructura base de una sección semántica.
    """

    return {
        "id": normalizar_identificador(
            titulo
        ),
        "titulo": limpiar_texto(
            titulo
        ),
        "tipo": tipo,
        "descripcion": "",
        "elementos": []
    }


# ----------------------------------------------------------
# Crear elemento documental
# ----------------------------------------------------------

def crear_elemento(
    titulo="",
    descripcion="",
    url="",
    tipo="enlace"
):
    """
    Crea una estructura homogénea para los elementos
    extraídos de las distintas secciones.
    """

    return {
        "tipo": tipo,
        "titulo": limpiar_texto(
            titulo
        ),
        "descripcion": limpiar_texto(
            descripcion
        ),
        "url": normalizar_url(
            url
        )
    }


# ==========================================================
# 5. ESTRUCTURA BASE DEL JSON
#
# Esta estructura representa la organización semántica
# de UPV Innovación y no la estructura técnica del HTML.
#
# La extracción del siguiente bloque rellenará estos campos.
# ==========================================================

def crear_json_base():
    """
    Construye la estructura inicial del JSON de Innovación.
    """

    return {

        # --------------------------------------------------
        # Información general de la página
        # --------------------------------------------------

        "titulo": "UPV Innovación",

        "url": URL_RAIZ,

        "tipo": "padre",

        # --------------------------------------------------
        # Secciones semánticas
        # --------------------------------------------------

        "secciones": [

            crear_seccion(
                "Presentación",
                "presentacion"
            ),

            crear_seccion(
                "Servicios de Innovación",
                "servicios"
            ),

            crear_seccion(
                "Iniciativas",
                "iniciativas"
            ),

            crear_seccion(
                "Historias de Innovación",
                "historias_innovacion"
            ),

            crear_seccion(
                "Temas",
                "temas"
            ),

            crear_seccion(
                "Explora UPV",
                "explora"
            ),

            crear_seccion(
                "Destacados",
                "destacados"
            )
        ]
    }


# ==========================================================
# 6. CREAR JSON INICIAL
# ==========================================================

json_innovacion = crear_json_base()


# ==========================================================
# 7. COMPROBACIÓN DE CONFIGURACIÓN
# ==========================================================

print()
print("=" * 70)
print("CONFIGURACIÓN DE UPV INNOVACIÓN")
print("=" * 70)

print()
print("Directorio del proyecto:")
print(ruta_programa)

print()
print("URL raíz:")
print(URL_RAIZ)

print()
print("JSON:")
print(RUTA_JSON)

print()
print("Carpeta JSON:")
print(CARPETA_JSON)

print()
print("Secciones semánticas:")

for seccion in json_innovacion["secciones"]:

    print(
        f"  - {seccion['titulo']}"
    )

print()
print("Estructura JSON inicial preparada.")

print()
print("=" * 70)


CONFIGURACIÓN DE UPV INNOVACIÓN

Directorio del proyecto:
/content/drive/MyDrive/TFG Teleco

URL raíz:
https://innovacion.upv.es/

JSON:
/content/drive/MyDrive/TFG Teleco/JSONs/innovacion.json

Carpeta JSON:
/content/drive/MyDrive/TFG Teleco/JSONs

Secciones semánticas:
  - Presentación
  - Servicios de Innovación
  - Iniciativas
  - Historias de Innovación
  - Temas
  - Explora UPV
  - Destacados

Estructura JSON inicial preparada.



In [4]:
# ==========================================================
# BLOQUE 4. EXTRACCIÓN Y GENERACIÓN DEL JSON
#
# Extrae la página raíz de UPV Innovación y construye una
# representación JSON basada en su estructura semántica.
#
# Flujo:
#
# HTML
#   ↓
# BeautifulSoup
#   ↓
# Identificación de bloques semánticos
#   ↓
# Extracción de contenido
#   ↓
# Normalización
#   ↓
# Deduplicación
#   ↓
# JSON
#
# No genera Markdown.
# ==========================================================

import requests
from bs4 import BeautifulSoup


# ==========================================================
# 1. DESCARGA DE LA PÁGINA
# ==========================================================

print()
print("=" * 70)
print("DESCARGA DE UPV INNOVACIÓN")
print("=" * 70)

print()
print("URL:")
print(URL_RAIZ)


try:

    respuesta = requests.get(
        URL_RAIZ,
        headers=HEADERS,
        timeout=30
    )

    respuesta.raise_for_status()

except requests.RequestException as error:

    print()
    print("ERROR AL DESCARGAR LA PÁGINA:")
    print(error)

    raise


print()
print("Código HTTP:")
print(respuesta.status_code)

print()
print("URL final:")
print(respuesta.url)

print()
print("Content-Type:")
print(
    respuesta.headers.get(
        "Content-Type",
        ""
    )
)

print()
print("Tamaño:")
print(
    len(
        respuesta.content
    ),
    "bytes"
)


# ==========================================================
# 2. PARSEAR HTML
# ==========================================================

soup = BeautifulSoup(
    respuesta.text,
    "html.parser"
)


# ==========================================================
# 3. FUNCIONES AUXILIARES DE EXTRACCIÓN
# ==========================================================

def extraer_texto(elemento):

    """
    Extrae el texto visible de un elemento HTML
    eliminando espacios redundantes.
    """

    if elemento is None:
        return ""

    return limpiar_texto(
        elemento.get_text(
            " ",
            strip=True
        )
    )


def extraer_url(elemento):

    """
    Extrae y normaliza la URL de un enlace.
    """

    if elemento is None:
        return ""

    url = elemento.get(
        "href",
        ""
    )

    return normalizar_url(
        url,
        respuesta.url
    )


def extraer_imagen(elemento):

    """
    Extrae la primera imagen encontrada dentro
    de un elemento.
    """

    if elemento is None:
        return ""

    imagen = elemento.find(
        "img"
    )

    if imagen is None:
        return ""

    src = (
        imagen.get("src")
        or imagen.get("data-src")
        or ""
    )

    return normalizar_url(
        src,
        respuesta.url
    )


def extraer_enlace_principal(elemento):

    """
    Obtiene el primer enlace relevante de un bloque.
    """

    if elemento is None:
        return ""

    enlace = elemento.find(
        "a",
        href=True
    )

    if enlace is None:
        return ""

    return extraer_url(
        enlace
    )


def buscar_seccion_por_tipo(
    json_data,
    tipo
):

    """
    Devuelve una sección del JSON a partir de su tipo.
    """

    for seccion in json_data["secciones"]:

        if seccion["tipo"] == tipo:

            return seccion

    return None


# ==========================================================
# 4. IDENTIFICAR CONTENEDOR PRINCIPAL
# ==========================================================

main = soup.find(
    "main"
)

if main is None:

    main = soup.body


if main is None:

    raise RuntimeError(
        "No se ha podido identificar "
        "el contenido principal de la página."
    )


# ==========================================================
# 5. INFORMACIÓN GENERAL
# ==========================================================

titulo_html = soup.find(
    "h1"
)

if titulo_html:

    json_innovacion["titulo"] = (
        extraer_texto(
            titulo_html
        )
        or json_innovacion["titulo"]
    )


json_innovacion["url"] = (
    respuesta.url
)


# ==========================================================
# 6. PRESENTACIÓN
# ==========================================================

seccion = buscar_seccion_por_tipo(
    json_innovacion,
    "presentacion"
)

if seccion:

    # Buscar el primer bloque principal de presentación

    bloque = main.find(
        class_=lambda clases:
        clases and any(
            "slider-ppal-home" in clase
            for clase in (
                clases
                if isinstance(clases, list)
                else [clases]
            )
        )
    )

    if bloque:

        seccion["descripcion"] = (
            extraer_texto(
                bloque
            )
        )

        seccion["elementos"] = []


# ==========================================================
# 7. SERVICIOS DE INNOVACIÓN
# ==========================================================

seccion = buscar_seccion_por_tipo(
    json_innovacion,
    "servicios"
)

if seccion:

    elementos = []

    bloques = main.select(
        ".vtm-servicios-grid__servicio"
    )

    # Si la clase específica no aparece,
    # se utiliza el patrón de blurb de Divi.

    if not bloques:

        bloques = main.select(
            ".vtm-servicios-grid .et_pb_blurb"
        )

    for bloque in bloques:

        titulo = ""

        titulo_elemento = bloque.find(
            class_=lambda clases:
            clases and (
                "et_pb_module_header" in clases
                if isinstance(clases, list)
                else "et_pb_module_header" in clases
            )
        )

        if titulo_elemento:

            titulo = extraer_texto(
                titulo_elemento
            )

        enlace = extraer_enlace_principal(
            bloque
        )

        descripcion = extraer_texto(
            bloque
        )

        if titulo:

            elementos.append({

                "tipo": "servicio",

                "titulo": titulo,

                "descripcion": descripcion,

                "url": enlace

            })

    seccion["elementos"] = (
        deduplicar_lista(
            elementos,
            clave="url"
        )
    )


# ==========================================================
# 8. INICIATIVAS
# ==========================================================

seccion = buscar_seccion_por_tipo(
    json_innovacion,
    "iniciativas"
)

if seccion:

    elementos = []

    bloques = main.select(
        ".card-iniciativa"
    )

    for bloque in bloques:

        titulo = ""

        titulo_elemento = bloque.find(
            [
                "h2",
                "h3",
                "h4"
            ]
        )

        if titulo_elemento:

            titulo = extraer_texto(
                titulo_elemento
            )

        enlace = extraer_enlace_principal(
            bloque
        )

        imagen = extraer_imagen(
            bloque
        )

        descripcion = extraer_texto(
            bloque
        )

        if titulo or enlace:

            elementos.append({

                "tipo": "iniciativa",

                "titulo": titulo,

                "descripcion": descripcion,

                "url": enlace,

                "imagen": imagen

            })

    seccion["elementos"] = (
        deduplicar_lista(
            elementos,
            clave="url"
        )
    )


# ==========================================================
# 9. HISTORIAS DE INNOVACIÓN
# ==========================================================

seccion = buscar_seccion_por_tipo(
    json_innovacion,
    "historias_innovacion"
)

if seccion:

    elementos = []

    bloques = main.select(
        ".card-historia-innovacion"
    )

    for bloque in bloques:

        titulo = ""

        titulo_elemento = bloque.find(
            [
                "h2",
                "h3",
                "h4"
            ]
        )

        if titulo_elemento:

            titulo = extraer_texto(
                titulo_elemento
            )

        enlace = extraer_enlace_principal(
            bloque
        )

        imagen = extraer_imagen(
            bloque
        )

        descripcion = extraer_texto(
            bloque
        )

        if titulo or enlace:

            elementos.append({

                "tipo": "historia",

                "titulo": titulo,

                "descripcion": descripcion,

                "url": enlace,

                "imagen": imagen

            })

    seccion["elementos"] = (
        deduplicar_lista(
            elementos,
            clave="url"
        )
    )


# ==========================================================
# 10. TEMAS
# ==========================================================

seccion = buscar_seccion_por_tipo(
    json_innovacion,
    "temas"
)

if seccion:

    elementos = []

    bloques = main.select(
        ".card-area-mini"
    )

    for bloque in bloques:

        titulo = ""

        titulo_elemento = bloque.find(
            [
                "h2",
                "h3",
                "h4"
            ]
        )

        if titulo_elemento:

            titulo = extraer_texto(
                titulo_elemento
            )

        enlace = extraer_enlace_principal(
            bloque
        )

        imagen = extraer_imagen(
            bloque
        )

        if titulo or enlace:

            elementos.append({

                "tipo": "tema",

                "titulo": titulo,

                "descripcion": "",

                "url": enlace,

                "imagen": imagen

            })

    seccion["elementos"] = (
        deduplicar_lista(
            elementos,
            clave="url"
        )
    )


# ==========================================================
# 11. EXPLORA UPV
# ==========================================================

seccion = buscar_seccion_por_tipo(
    json_innovacion,
    "explora"
)

if seccion:

    bloque = main.select_one(
        ".c2action-explora"
    )

    if bloque:

        titulo = ""

        titulo_elemento = bloque.find(
            [
                "h2",
                "h3",
                "h4"
            ]
        )

        if titulo_elemento:

            titulo = extraer_texto(
                titulo_elemento
            )

        seccion["descripcion"] = (
            extraer_texto(
                bloque
            )
        )

        seccion["elementos"] = [

            {

                "tipo": "recurso",

                "titulo": titulo,

                "descripcion": extraer_texto(
                    bloque
                ),

                "url": extraer_enlace_principal(
                    bloque
                )

            }

        ]


# ==========================================================
# 12. DESTACADOS
# ==========================================================

seccion = buscar_seccion_por_tipo(
    json_innovacion,
    "destacados"
)

if seccion:

    elementos = []

    bloques = main.select(
        ".slider-destacados-home__item"
    )

    for bloque in bloques:

        titulo = ""

        titulo_elemento = bloque.find(
            [
                "h2",
                "h3",
                "h4"
            ]
        )

        if titulo_elemento:

            titulo = extraer_texto(
                titulo_elemento
            )

        enlace = extraer_enlace_principal(
            bloque
        )

        imagen = extraer_imagen(
            bloque
        )

        descripcion = extraer_texto(
            bloque
        )

        if titulo or enlace:

            elementos.append({

                "tipo": "destacado",

                "titulo": titulo,

                "descripcion": descripcion,

                "url": enlace,

                "imagen": imagen

            })

    seccion["elementos"] = (
        deduplicar_lista(
            elementos,
            clave="url"
        )
    )


# ==========================================================
# 13. LIMPIEZA FINAL DEL JSON
# ==========================================================

for seccion in json_innovacion["secciones"]:

    seccion["titulo"] = limpiar_texto(
        seccion.get(
            "titulo",
            ""
        )
    )

    seccion["descripcion"] = limpiar_texto(
        seccion.get(
            "descripcion",
            ""
        )
    )

    if "elementos" not in seccion:

        seccion["elementos"] = []


# ==========================================================
# 14. GUARDAR JSON
# ==========================================================

with open(
    RUTA_JSON,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        json_innovacion,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ==========================================================
# 15. RESUMEN DE LA EXTRACCIÓN
# ==========================================================

print()
print("=" * 70)
print("JSON DE UPV INNOVACIÓN GENERADO")
print("=" * 70)

print()
print("Fichero:")
print(RUTA_JSON)

print()
print("Título:")
print(
    json_innovacion["titulo"]
)

print()
print("Secciones:")

for seccion in json_innovacion["secciones"]:

    print(
        f"  - {seccion['titulo']}: "
        f"{len(seccion['elementos'])} elementos"
    )

print()
print(
    "JSON generado correctamente."
)

print("=" * 70)


DESCARGA DE UPV INNOVACIÓN

URL:
https://innovacion.upv.es/

Código HTTP:
200

URL final:
https://innovacion.upv.es/

Content-Type:
text/html; charset=UTF-8

Tamaño:
187958 bytes

JSON DE UPV INNOVACIÓN GENERADO

Fichero:
/content/drive/MyDrive/TFG Teleco/JSONs/innovacion.json

Título:
Conectamos sociedad, empresa y universidad para innovar juntos →

Secciones:
  - Presentación: 0 elementos
  - Servicios de Innovación: 4 elementos
  - Iniciativas: 5 elementos
  - Historias de Innovación: 6 elementos
  - Temas: 9 elementos
  - Explora UPV: 1 elementos
  - Destacados: 4 elementos

JSON generado correctamente.


In [6]:
# ==========================================================
# BLOQUE 13. GENERACIÓN DE FICHEROS MARKDOWN
#
# Estructura:
#
# INVESTIGACION/
# └── INNOVACION/
#     ├── innovacion.md
#     └── recursos/
#         ├── talento.md
#         ├── formacion.md
#         ├── idi.md
#         └── ...
#
# El Markdown de cada recurso se genera visitando
# realmente la URL almacenada en el JSON.
#
# ==========================================================

import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


# ==========================================================
# 1. RUTAS
# ==========================================================

CARPETA_INVESTIGACION = os.path.join(
    ruta_programa,
    "INVESTIGACION"
)

CARPETA_INNOVACION = os.path.join(
    CARPETA_INVESTIGACION,
    "INNOVACION"
)

CARPETA_RECURSOS = os.path.join(
    CARPETA_INNOVACION,
    "recursos"
)

os.makedirs(
    CARPETA_RECURSOS,
    exist_ok=True
)

RUTA_MARKDOWN_PADRE = os.path.join(
    CARPETA_INNOVACION,
    "innovacion.md"
)


# ==========================================================
# 2. CABECERAS HTTP
# ==========================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0 Safari/537.36"
    )
}


# ==========================================================
# 3. FUNCIONES AUXILIARES
# ==========================================================

def limpiar_texto(texto):

    if not texto:
        return ""

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()


def normalizar_url(url, url_base=None):

    if not url:
        return ""

    url = url.strip()

    if url_base:
        url = urljoin(
            url_base,
            url
        )

    return url


def nombre_archivo_markdown(titulo, url=""):

    nombre = titulo.strip()

    if not nombre:
        nombre = urlparse(url).path.strip("/").split("/")[-1]

    if not nombre:
        nombre = "recurso"

    reemplazos = {
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
        "ñ": "n"
    }

    nombre = nombre.lower()

    for origen, destino in reemplazos.items():

        nombre = nombre.replace(
            origen,
            destino
        )

    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )

    nombre = nombre.strip("_")

    if not nombre:
        nombre = "recurso"

    return nombre + ".md"


def escapar_yaml(texto):

    if texto is None:
        return ""

    texto = str(texto)

    return texto.replace(
        '"',
        '\\"'
    )


def eliminar_elementos_no_informativos(soup):

    elementos = [
        "script",
        "style",
        "noscript",
        "iframe",
        "svg",
        "form"
    ]

    for etiqueta in elementos:

        for elemento in soup.find_all(etiqueta):

            elemento.decompose()

    return soup


def extraer_contenido_pagina(url):

    """
    Descarga una página y extrae su contenido textual
    principal conservando una estructura básica de
    títulos, párrafos, listas y tablas.

    Devuelve:

        titulo
        contenido_markdown
        url_final

    o None si la página no puede procesarse.
    """

    try:

        respuesta = requests.get(
            url,
            headers=HEADERS,
            timeout=30,
            allow_redirects=True
        )

        respuesta.raise_for_status()

    except requests.RequestException as error:

        print(
            f"    ERROR descargando: {error}"
        )

        return None


    content_type = respuesta.headers.get(
        "Content-Type",
        ""
    ).lower()


    if "text/html" not in content_type:

        print(
            "    DESCARTADO: no es HTML"
        )

        return None


    soup = BeautifulSoup(
        respuesta.text,
        "html.parser"
    )

    soup = eliminar_elementos_no_informativos(
        soup
    )


    # ------------------------------------------------------
    # Título
    # ------------------------------------------------------

    titulo = ""

    if soup.title:

        titulo = limpiar_texto(
            soup.title.get_text(
                " ",
                strip=True
            )
        )


    # ------------------------------------------------------
    # Intentar localizar el contenido principal
    # ------------------------------------------------------

    contenido_principal = None

    selectores = [
        "main",
        "article",
        "[role='main']",
        ".entry-content",
        ".post-content",
        ".page-content",
        ".elementor-widget-theme-post-content"
    ]

    for selector in selectores:

        elemento = soup.select_one(
            selector
        )

        if elemento:

            contenido_principal = elemento

            break


    if contenido_principal is None:

        contenido_principal = soup.body


    if contenido_principal is None:

        return None


    # ------------------------------------------------------
    # Eliminar elementos de navegación/interfaz
    # ------------------------------------------------------

    for selector in [
        "header",
        "footer",
        "nav",
        ".menu",
        ".navbar",
        ".navigation",
        ".breadcrumb",
        ".breadcrumbs",
        ".cookie",
        ".cookies"
    ]:

        for elemento in contenido_principal.select(
            selector
        ):

            elemento.decompose()


    # ------------------------------------------------------
    # Construcción Markdown
    # ------------------------------------------------------

    bloques = []


    for elemento in contenido_principal.find_all(
        [
            "h1",
            "h2",
            "h3",
            "h4",
            "p",
            "ul",
            "ol",
            "table"
        ]
    ):

        nombre = elemento.name


        # --------------------------------------------------
        # Encabezados
        # --------------------------------------------------

        if nombre in [
            "h1",
            "h2",
            "h3",
            "h4"
        ]:

            texto = limpiar_texto(
                elemento.get_text(
                    " ",
                    strip=True
                )
            )

            if texto:

                nivel = int(
                    nombre[1]
                )

                bloques.append(
                    "#" * nivel
                    + " "
                    + texto
                )


        # --------------------------------------------------
        # Párrafos
        # --------------------------------------------------

        elif nombre == "p":

            texto = limpiar_texto(
                elemento.get_text(
                    " ",
                    strip=True
                )
            )

            if texto:

                bloques.append(
                    texto
                )


        # --------------------------------------------------
        # Listas
        # --------------------------------------------------

        elif nombre in [
            "ul",
            "ol"
        ]:

            items = []

            for li in elemento.find_all(
                "li",
                recursive=False
            ):

                texto = limpiar_texto(
                    li.get_text(
                        " ",
                        strip=True
                    )
                )

                if texto:

                    items.append(
                        f"- {texto}"
                    )

            if items:

                bloques.append(
                    "\n".join(items)
                )


        # --------------------------------------------------
        # Tablas
        # --------------------------------------------------

        elif nombre == "table":

            filas = []

            for tr in elemento.find_all(
                "tr"
            ):

                celdas = []

                for celda in tr.find_all(
                    ["th", "td"]
                ):

                    texto = limpiar_texto(
                        celda.get_text(
                            " ",
                            strip=True
                        )
                    )

                    celdas.append(
                        texto
                    )

                if celdas:

                    filas.append(
                        celdas
                    )

            if filas:

                numero_columnas = max(
                    len(fila)
                    for fila in filas
                )

                encabezado = filas[0]

                encabezado += [
                    ""
                ] * (
                    numero_columnas
                    - len(encabezado)
                )

                markdown = []

                markdown.append(
                    "| "
                    + " | ".join(
                        encabezado
                    )
                    + " |"
                )

                markdown.append(
                    "| "
                    + " | ".join(
                        ["---"] * numero_columnas
                    )
                    + " |"
                )

                for fila in filas[1:]:

                    fila += [
                        ""
                    ] * (
                        numero_columnas
                        - len(fila)
                    )

                    markdown.append(
                        "| "
                        + " | ".join(
                            fila
                        )
                        + " |"
                    )

                bloques.append(
                    "\n".join(markdown)
                )


    contenido = "\n\n".join(
        bloques
    ).strip()


    if not contenido:

        return None


    return {
        "titulo": titulo,
        "contenido": contenido,
        "url_final": respuesta.url
    }


# ==========================================================
# 4. GENERACIÓN DEL MARKDOWN DE LA PÁGINA PADRE
# ==========================================================

print()
print("=" * 70)
print("GENERACIÓN DE MARKDOWN DE INNOVACIÓN")
print("=" * 70)


# ----------------------------------------------------------
# Cargar JSON generado anteriormente
# ----------------------------------------------------------

with open(
    RUTA_JSON,
    "r",
    encoding="utf-8"
) as archivo:

    datos = json.load(
        archivo
    )


# ----------------------------------------------------------
# Markdown principal
# ----------------------------------------------------------

titulo_padre = datos.get(
    "titulo",
    "UPV Innovación"
)

url_padre = datos.get(
    "url",
    URL_RAIZ
)


with open(
    RUTA_MARKDOWN_PADRE,
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(
        "---\n"
    )

    archivo.write(
        "tipo: padre\n"
    )

    archivo.write(
        "nivel: innovacion\n"
    )

    archivo.write(
        f"url: {url_padre}\n"
    )

    archivo.write(
        "---\n\n"
    )

    archivo.write(
        f"# {titulo_padre}\n\n"
    )


    for seccion in datos.get(
        "secciones",
        []
    ):

        titulo_seccion = limpiar_texto(
            seccion.get(
                "titulo",
                ""
            )
        )

        if titulo_seccion:

            archivo.write(
                f"## {titulo_seccion}\n\n"
            )


        descripcion = limpiar_texto(
            seccion.get(
                "descripcion",
                ""
            )
        )

        if descripcion:

            archivo.write(
                descripcion
                + "\n\n"
            )


        elementos = seccion.get(
            "elementos",
            []
        )


        for elemento in elementos:

            titulo_elemento = limpiar_texto(
                elemento.get(
                    "titulo",
                    ""
                )
            )

            descripcion_elemento = limpiar_texto(
                elemento.get(
                    "descripcion",
                    ""
                )
            )

            url = normalizar_url(
                elemento.get(
                    "url",
                    ""
                ),
                url_padre
            )


            if titulo_elemento:

                archivo.write(
                    f"### {titulo_elemento}\n\n"
                )

            elif elemento.get(
                "tipo"
            ):

                archivo.write(
                    f"### "
                    f"{elemento.get('tipo').capitalize()}"
                    f"\n\n"
                )


            if descripcion_elemento:

                archivo.write(
                    descripcion_elemento
                    + "\n\n"
                )


            if url:

                archivo.write(
                    f"[{url}]({url})\n\n"
                )


print(
    "Markdown principal generado:"
)

print(
    RUTA_MARKDOWN_PADRE
)


# ==========================================================
# 5. RECORRER TODAS LAS URL DEL JSON
# ==========================================================

print()
print("=" * 70)
print("GENERACIÓN DE RECURSOS")
print("=" * 70)


recursos_generados = 0
recursos_descartados = 0
urls_procesadas = set()


for seccion in datos.get(
    "secciones",
    []
):

    nombre_seccion = limpiar_texto(
        seccion.get(
            "titulo",
            ""
        )
    )

    for elemento in seccion.get(
        "elementos",
        []
    ):

        url = normalizar_url(
            elemento.get(
                "url",
                ""
            ),
            url_padre
        )


        # --------------------------------------------------
        # Sin URL
        # --------------------------------------------------

        if not url:

            continue


        # --------------------------------------------------
        # Evitar duplicados
        # --------------------------------------------------

        if url in urls_procesadas:

            continue

        urls_procesadas.add(
            url
        )


        # --------------------------------------------------
        # Solo páginas HTTP/HTTPS
        # --------------------------------------------------

        if not url.startswith(
            (
                "http://",
                "https://"
            )
        ):

            continue


        print()
        print(
            f"[{recursos_generados + recursos_descartados + 1}] "
            f"{url}"
        )


        # --------------------------------------------------
        # Descargar página real
        # --------------------------------------------------

        resultado = extraer_contenido_pagina(
            url
        )


        if resultado is None:

            recursos_descartados += 1

            continue


        titulo_recurso = (
            limpiar_texto(
                elemento.get(
                    "titulo",
                    ""
                )
            )
            or
            resultado.get(
                "titulo",
                ""
            )
        )


        # --------------------------------------------------
        # Si el título HTML es genérico, utilizar
        # el nombre de la página como último recurso
        # --------------------------------------------------

        if not titulo_recurso:

            titulo_recurso = (
                urlparse(
                    resultado["url_final"]
                ).path
                .strip("/")
                .split("/")[-1]
                or "Recurso"
            )


        nombre_archivo = nombre_archivo_markdown(
            titulo_recurso,
            resultado["url_final"]
        )


        ruta_recurso = os.path.join(
            CARPETA_RECURSOS,
            nombre_archivo
        )


        # --------------------------------------------------
        # Metadatos
        # --------------------------------------------------

        tipo_elemento = elemento.get(
            "tipo",
            "recurso"
        )


        # Normalizar categoría del recurso
        #
        # Los valores originales del JSON
        # (servicio, iniciativa, historia, tema, etc.)
        # se conservan como tipo_recurso.

        tipo_recurso = tipo_elemento


        # --------------------------------------------------
        # Generar Markdown
        # --------------------------------------------------

        with open(
            ruta_recurso,
            "w",
            encoding="utf-8"
        ) as archivo:

            archivo.write(
                "---\n"
            )

            archivo.write(
                "tipo: recurso\n"
            )

            archivo.write(
                "nivel: innovacion\n"
            )

            archivo.write(
                f"tipo_recurso: "
                f"{escapar_yaml(tipo_recurso)}\n"
            )

            archivo.write(
                f"seccion_origen: "
                f"{escapar_yaml(nombre_seccion)}\n"
            )

            archivo.write(
                f"url: "
                f"{escapar_yaml(resultado['url_final'])}\n"
            )

            archivo.write(
                f"url_origen: "
                f"{escapar_yaml(url_padre)}\n"
            )

            archivo.write(
                "---\n\n"
            )


            # ----------------------------------------------
            # Título real de la página
            # ----------------------------------------------

            archivo.write(
                f"# {titulo_recurso}\n\n"
            )


            # ----------------------------------------------
            # Contenido REAL obtenido visitando la URL
            # ----------------------------------------------

            archivo.write(
                resultado["contenido"]
            )

            archivo.write(
                "\n"
            )


        recursos_generados += 1

        print(
            f"    OK -> {nombre_archivo}"
        )


# ==========================================================
# 6. RESUMEN FINAL
# ==========================================================

print()
print("=" * 70)
print("GENERACIÓN COMPLETADA")
print("=" * 70)

print()
print(
    "Directorio principal:"
)

print(
    CARPETA_INNOVACION
)

print()
print(
    "Markdown de página padre:"
)

print(
    RUTA_MARKDOWN_PADRE
)

print()
print(
    "URLs detectadas:"
)

print(
    len(urls_procesadas)
)

print()
print(
    "Recursos Markdown generados:"
)

print(
    recursos_generados
)

print()
print(
    "Recursos descartados:"
)

print(
    recursos_descartados
)

print()
print(
    "Estructura generada:"
)

print(
    "INVESTIGACION/"
)

print(
    "└── INNOVACION/"
)

print(
    "    ├── innovacion.md"
)

print(
    "    └── recursos/"
)

for nombre in sorted(
    os.listdir(
        CARPETA_RECURSOS
    )
):

    if nombre.endswith(
        ".md"
    ):

        print(
            f"        ├── {nombre}"
        )


GENERACIÓN DE MARKDOWN DE INNOVACIÓN
Markdown principal generado:
/content/drive/MyDrive/TFG Teleco/INVESTIGACION/INNOVACION/innovacion.md

GENERACIÓN DE RECURSOS

[1] https://innovacion.upv.es/talento/
    OK -> talento.md

[2] https://innovacion.upv.es/formacion/
    OK -> formacion.md

[3] https://innovacion.upv.es/idi/
    OK -> i_d_i.md

[4] https://innovacion.upv.es/emprendimiento/
    OK -> emprendimiento.md

[5] https://innovacion.upv.es/iniciativas/matchmaking/match-upv/
    OK -> match_upv_upv_innovacion.md

[6] https://innovacion.upv.es/iniciativas/jornadas-upv-innovacion/forum-upv-innovacion-2026-primavera-side-event/
    OK -> forum_upv_innovacion_2026_primavera_side_event_upv_innovacion.md

[7] https://innovacion.upv.es/iniciativas/cpi-tour-experience/cpi-tour-experience-para-estudiantes-upv-piae-febrero-2026/
    OK -> cpi_tour_experience_para_estudiantes_upv_piae_febrero_2026_upv_innovacion.md

[8] https://innovacion.upv.es/iniciativas/spin-upv/
    OK -> spin_upv_upv_